In [1]:
import torch
import torch.nn as nn
import numpy as np

# 1. Prepare the Data
text = "pytorch"
chars = sorted(list(set(text)))
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

vocab_size = len(chars)
input_size = vocab_size
hidden_size = 16
output_size = vocab_size

# Create training sequences (Inputs: "pytorc", Targets: "ytorch")
inputs = [char_to_ix[ch] for ch in text[:-1]]
targets = [char_to_ix[ch] for ch in text[1:]]

# Convert inputs to One-Hot encoding vectors
def to_one_hot(indices, vocab_size):
    one_hot = np.zeros((len(indices), vocab_size), dtype=np.float32)
    for i, idx in enumerate(indices):
        one_hot[i][idx] = 1.0
    return torch.tensor(one_hot)

input_tensors = to_one_hot(inputs, vocab_size).unsqueeze(1) # Shape: [seq_len, batch_size, input_size]
target_tensors = torch.tensor(targets, dtype=torch.long)

# 2. Define the RNN Model
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=1)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, hidden):
        # x shape: [seq_len, batch_size, input_size]
        out, hidden = self.rnn(x, hidden)
        # out shape: [seq_len, batch_size, hidden_size]
        out = self.fc(out.squeeze(1)) # Collapse batch dim for Linear layer
        return out, hidden

    def init_hidden(self):
        return torch.zeros(1, 1, self.hidden_size)

# Initialize Model, Loss, and Optimizer
model = SimpleRNN(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

# 3. Training Loop
epochs = 100
for epoch in range(epochs):
    hidden = model.init_hidden()
    optimizer.zero_grad()
    
    output, hidden = model(input_tensors, hidden)
    loss = criterion(output, target_tensors)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# 4. Testing / Inference
print("\nTesting the model:")
model.eval()
with torch.no_grad():
    hidden = model.init_hidden()
    current_char = "p"
    result = current_char
    
    for _ in range(len(text) - 1):
        idx = char_to_ix[current_char]
        input_tensor = to_one_hot([idx], vocab_size).unsqueeze(1)
        
        output, hidden = model(input_tensor, hidden)
        pred_idx = torch.argmax(output, dim=1).item()
        
        current_char = ix_to_char[pred_idx]
        result += current_char

    print(f"Input: 'p' -> Predicted Sequence: '{result}'")


Epoch [20/100], Loss: 0.0006
Epoch [40/100], Loss: 0.0001
Epoch [60/100], Loss: 0.0001
Epoch [80/100], Loss: 0.0001
Epoch [100/100], Loss: 0.0001

Testing the model:
Input: 'p' -> Predicted Sequence: 'pytorch'
